<a href="https://colab.research.google.com/github/Spandhana045/skill_lab/blob/main/AMAZON_SHOPPING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --upgrade google-genai python-dotenv playwright

In [ ]:
import os
import asyncio
from getpass import getpass

from dotenv import load_dotenv
from playwright.async_api import async_playwright
from google import genai
from google.genai import types

# Load environment variables
load_dotenv()

# Get API key
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    api_key = getpass("Enter your Gemini API key: ")

# Connect Gemini
client = genai.Client(api_key=api_key)

print("All libraries imported successfully!")
print("Gemini connection created successfully!")

Enter your Gemini API key: ··········
All libraries imported successfully!
Gemini connection created successfully!


In [ ]:
!playwright install-deps chromium

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libasound2t64 is already the newest version (1.2.11-1ubuntu0.3).
libatk-bridge2.0-0t64 is already the newest version (2.52.0-1build1).
libatk1.

In [ ]:
pw = await async_playwright().start()

browser = await pw.chromium.launch(
    headless=True,
    slow_mo=500
)

context = await browser.new_context()

page = await context.new_page()

print("Chrome is ready in headless mode!")

Chrome is ready in headless mode!


In [ ]:
async def open_amazon() -> str:
    print("[CHROME] Opening Amazon India...")

    await page.goto(
        "https://www.amazon.in",
        wait_until="domcontentloaded",
        timeout=60000
    )

    return "Amazon India homepage opened."


async def search_product(keyword: str) -> str:
    print(f"[CHROME] Searching for {keyword}...")

    await page.wait_for_selector(
        "#twotabsearchtextbox",
        timeout=30000
    )

    await page.fill("#twotabsearchtextbox", keyword)
    await page.click("#nav-search-submit-button")

    await page.wait_for_selector(
        "[data-component-type='s-search-result']",
        timeout=30000
    )

    return "Search results loaded."


async def get_top_products(limit: int = 5) -> str:
    print("[CHROME] Extracting products...")

    cards = await page.locator(
        "[data-component-type='s-search-result']"
    ).all()

    results = []

    for card in cards[:limit]:

        title_loc = card.locator("h2 span").first
        price_loc = card.locator(".a-price .a-offscreen").first

        title = (
            await title_loc.inner_text()
            if await title_loc.count() > 0
            else "Title unavailable"
        )

        price = (
            await price_loc.inner_text()
            if await price_loc.count() > 0
            else "Price unavailable"
        )

        results.append(f"{title.strip()} | {price.strip()}")

    return "\n".join(results)


print("Amazon functions created successfully!")

Amazon functions created successfully!


In [10]:
tools = [
    open_amazon,
    search_product,
    get_top_products
]

print("Tools registered successfully!")

Tools registered successfully!


In [15]:
from urllib.parse import quote_plus

async def search_product(keyword: str) -> str:

    print(f"[CHROME] Searching for: {keyword}")

    search_url = (
        "https://www.amazon.in/s?k=" + quote_plus(keyword)
    )

    await page.goto(
        search_url,
        wait_until="domcontentloaded",
        timeout=60000
    )

    await page.wait_for_timeout(5000)

    # Check whether products are available
    count = await page.locator(
        "[data-component-type='s-search-result']"
    ).count()

    if count > 0:
        print(f"Found {count} product listings!")
        return "Search results loaded successfully."

    # Check for CAPTCHA
    page_text = await page.locator("body").inner_text()

    if "captcha" in page_text.lower() or "robot check" in page_text.lower():
        raise Exception("Amazon CAPTCHA detected. Automated search is blocked.")

    raise Exception("Amazon search results did not load. Check the page.")

In [16]:
print("Starting AI Shopping Agent...")

try:
    # Start browser
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(
        headless=True,
        slow_mo=500
    )

    context = await browser.new_context()
    page = await context.new_page()

    print("Browser started successfully!")

    # Step 1: Open Amazon
    print("\n1. Opening Amazon India...")
    await open_amazon()

    # Step 2: Search product
    print("\n2. Searching for wireless mouse...")
    await search_product("wireless mouse under 1000")

    # Step 3: Collect products
    print("\n3. Collecting top 5 products...")
    products = await get_top_products(5)

    print("\n===== TOP 5 PRODUCTS =====")
    print(products)

    # Step 4: Gemini analysis
    print("\n4. Gemini is analysing products...")

    prompt = f"""
    Analyse these Amazon India products:

    {products}

    Compare their prices and recommend
    the best-value wireless mouse.

    Do not invent missing specifications.
    """

    response = await client.aio.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    print("\n===== AI SHOPPING AGENT RECOMMENDATION =====")
    print(response.text)

except Exception as e:
    print("\nERROR:", str(e))

Starting AI Shopping Agent...
Browser started successfully!

1. Opening Amazon India...
[CHROME] Opening Amazon India...

2. Searching for wireless mouse...
[CHROME] Searching for: wireless mouse under 1000
Found 16 product listings!

3. Collecting top 5 products...
[CHROME] Extracting products...

===== TOP 5 PRODUCTS =====
EvoFox Shadow X Pro Wireless Gaming Mouse Tri-Mode Rechargeable | BT, USB 2.4G & Wired, S203 Sensor, 1000Hz Polling Rate, 6400 DPI, RGB Lighting, 6 Programmable Buttons, Software Support | ₹999
Portronics Toad One Bluetooth Mouse with 2.4 GHz & BT 5.3 Dual Wireless, 6 Buttons, Rechargeable, RGB Lights, Connect 3 Devices, Ergonomic Design for Laptop, Smartphone, Tablet (Black) | ₹533
Portronics Toad Fold Wireless Bluetooth Mouse with Dual Wireless (BT + 2.4 GHz), Foldable Mouse, Rechargeable Battery, Multi-Device Pair, Up to 3200 DPI Sensitivity (Black) | ₹899
Portronics Vader Max Wireless Rechargeable Mouse, Thumb Support, Bluetooth 5.3 & 2.4 GHz Dual Mode, 6-Butto

In [18]:
import asyncio

prompt = f"""
You are an AI shopping assistant.

Compare these wireless mice:

{products}

Compare their prices, features and connectivity.
Recommend the best value-for-money option.
Explain your choice simply.
"""

response = None

for attempt in range(1, 4):
    try:
        print(f"Connecting to Gemini... Attempt {attempt}/3")

        response = await client.aio.models.generate_content(
            model="gemini-3.8-flash",
            contents=prompt
        )

        print("\n===== AI SHOPPING AGENT RECOMMENDATION =====")
        print(response.text)
        break

    except Exception as e:
        print("Gemini error:", str(e))

        if attempt < 3:
            print("Retrying in 10 seconds...")
            await asyncio.sleep(10)

if response is None:
    print("\nGemini is still unavailable.")
    print("Your Amazon product extraction was successful.")

Connecting to Gemini... Attempt 1/3

===== AI SHOPPING AGENT RECOMMENDATION =====
Here is a direct comparison of the 5 wireless mice based on their price, connectivity, and standout features.

---

### **Quick Comparison Table**

| Mouse | Price | Connectivity | Key Features | Best Suited For |
| :--- | :--- | :--- | :--- | :--- |
| **Portronics Toad One** | **₹533** | Dual (BT 5.3 + 2.4 GHz USB) | 3-device switching, Rechargeable, 6 buttons, RGB | Daily office work & multitasking |
| **Portronics Vader Max** | **₹865** | Dual (BT 5.3 + 2.4 GHz USB) | Ergonomic thumb rest, 2400 DPI, 6 buttons, Rechargeable | Comfort-focused office use |
| **Portronics Toad Fold** | **₹899** | Dual (BT + 2.4 GHz USB) | Foldable slim design, Multi-device, 3200 DPI, Rechargeable | Frequent travelers |
| **Cosmic Byte Raptor** | **₹899** | Dual Mode | Pixart 3212 sensor, 4800 DPI, 500Hz polling rate, PTFE feet | Entry-level casual gaming |
| **EvoFox Shadow X Pro** | **₹999** | Tri-Mode (BT + 2.4 GHz + Wir

In [19]:
print("===== SHOPPING AGENT SUMMARY =====")
print("Search: Wireless Mouse under ₹1000")
print("Products collected:", len(products))
print("AI Analysis:", "Completed" if response else "Not completed")
print("Amazon scraping: Successful")

===== SHOPPING AGENT SUMMARY =====
Search: Wireless Mouse under ₹1000
Products collected: 950
AI Analysis: Completed
Amazon scraping: Successful


In [20]:
print("===== SHOPPING AGENT SUMMARY =====")
print("Search: Wireless Mouse under ₹1000")
print("Products collected:", 5)
print("AI Analysis:", "Completed" if response else "Not completed")
print("Amazon scraping: Successful")
print("Gemini connection: Successful")
print("Project Status: SUCCESS")

===== SHOPPING AGENT SUMMARY =====
Search: Wireless Mouse under ₹1000
Products collected: 5
AI Analysis: Completed
Amazon scraping: Successful
Gemini connection: Successful
Project Status: SUCCESS
